# LeetCode 1047 — Remove All Adjacent Duplicates In String bằng Python

**Link đề:** https://leetcode.com/problems/remove-all-adjacent-duplicates-in-string/
**Độ khó:** Easy · **Chủ đề:** String, Stack (ngăn xếp)

```python
class Solution:
    def removeDuplicates(self, s: str) -> str:
        ...
```

## Mục tiêu học
1. Hiểu đúng thao tác "xóa cặp ký tự kề nhau giống nhau" và vì sao phải xóa **lặp đi lặp lại**.
2. Viết được cách trực giác (mô phỏng xóa nhiều lượt) và thấy vì sao nó chậm (`O(n²)`).
3. Hiểu cấu trúc dữ liệu **stack** (ngăn xếp) bằng `list` trong Python: `append`, `pop`, `[-1]`.
4. Nắm lời giải chính bằng stack `O(n)`, trace từng ký tự, chứng minh tính đúng.
5. Biết thêm cách **hai con trỏ ghi đè** (two pointers) và cách dùng `str.replace` lặp.
6. Tự giải các biến thể (xóa k ký tự liên tiếp, xóa cặp chữ hoa/thường…).

**Cách học:** đọc markdown → đoán output của cell code kế tiếp → chạy → so sánh. Các cell TODO chỉ để luyện, không tự chạy.

## 1. Đề bài — xóa cặp kề nhau giống nhau cho tới khi không xóa được nữa

Cho chuỗi `s` gồm chữ cái thường tiếng Anh. Một **lần xóa trùng** (duplicate removal) là chọn **hai ký tự kề nhau và giống nhau** rồi xóa cả hai.

Ta lặp lại việc xóa trên `s` **cho tới khi không còn cặp nào để xóa**. Trả về chuỗi cuối cùng. Người ta chứng minh được đáp án là **duy nhất** (không phụ thuộc thứ tự xóa).

| Input `s` | Output | Giải thích |
|---|---|---|
| `"abbaca"` | `"ca"` | xóa `bb` → `"aaca"`; xóa `aa` → `"ca"`; hết cặp |
| `"azxxzy"` | `"ay"` | xóa `xx` → `"azzy"`; xóa `zz` → `"ay"` |

**Constraints:**
- `1 <= s.length <= 10^5`
- `s` chỉ gồm chữ cái thường.

Điểm cần chú ý ngay: sau khi xóa một cặp, hai ký tự **vốn không kề nhau** có thể trở nên kề nhau (như `a` và `a` trong `"abbaca"` sau khi `bb` biến mất). Đó là "phản ứng dây chuyền" làm bài này thú vị.

In [1]:
# Xem "phản ứng dây chuyền" bằng tay: xóa thủ công từng bước bằng slicing
s = "abbaca"
print("ban đầu        :", s)
s = s[:1] + s[3:]      # bỏ "bb" ở vị trí 1, 2
print("sau khi xóa bb :", s)
s = s[:0] + s[2:]      # bỏ "aa" ở vị trí 0, 1
print("sau khi xóa aa :", s)

ban đầu        : abbaca
sau khi xóa bb : aaca
sau khi xóa aa : ca


## 2. Nền tảng Python cần biết

### 2.1. Chỉ số (index) và slicing trên chuỗi

- `s[i]` lấy ký tự thứ `i` (đếm từ 0). `s[-1]` lấy ký tự **cuối**.
- `s[a:b]` lấy đoạn từ `a` đến **trước** `b`. `s[:a]` là phần đầu, `s[b:]` là phần đuôi.
- Chuỗi Python là **bất biến** (immutable): không sửa tại chỗ được, mỗi lần "xóa" thực chất là **tạo chuỗi mới** → tốn `O(n)`.

In [2]:
s = "abbaca"
print(s[0], s[1], s[-1])     # a b a
print(s[:1])                 # 'a'
print(s[3:])                 # 'aca'
print(s[:1] + s[3:])         # 'aaca'  (bỏ s[1], s[2])

a b a
a
aca
aaca


### 2.2. `list` dùng như **stack** (ngăn xếp)

Stack giống **chồng đĩa**: chỉ được đặt đĩa lên **đỉnh** và lấy đĩa ở **đỉnh** (LIFO — Last In, First Out: vào sau ra trước).

Trong Python, `list` làm stack rất tốt:

| Thao tác | Cú pháp | Ý nghĩa | Chi phí |
|---|---|---|---|
| push | `stack.append(x)` | đặt `x` lên đỉnh | `O(1)` trung bình |
| pop | `stack.pop()` | lấy và xóa phần tử đỉnh | `O(1)` |
| peek | `stack[-1]` | xem đỉnh, không xóa | `O(1)` |
| rỗng? | `not stack` | `True` nếu stack rỗng | `O(1)` |

In [3]:
stack = []
stack.append("a")
stack.append("b")
stack.append("c")
print("stack:", stack)          # ['a', 'b', 'c']  -> đỉnh là 'c'
print("đỉnh :", stack[-1])      # c
top = stack.pop()
print("pop ra:", top, "| còn lại:", stack)
print("rỗng chưa?", not stack)  # False
stack.pop(); stack.pop()
print("rỗng chưa?", not stack)  # True

stack: ['a', 'b', 'c']
đỉnh : c
pop ra: c | còn lại: ['a', 'b']
rỗng chưa? False
rỗng chưa? True


### 2.3. Cẩn thận: `stack[-1]` trên stack rỗng gây lỗi

Nếu stack rỗng mà gọi `stack[-1]` → `IndexError`. Vì vậy luôn kiểm tra `stack and stack[-1] == ch`.
Toán tử `and` trong Python **ngắt mạch** (short-circuit): nếu vế trái `stack` là rỗng (falsy) thì vế phải **không được tính**, nên không lỗi.

In [4]:
stack = []
ch = "a"
print(bool(stack))                       # False: list rỗng là "falsy"
print(stack and stack[-1] == ch)         # [] -> không tính vế phải
try:
    stack[-1]
except IndexError as e:
    print("Lỗi khi đọc đỉnh stack rỗng:", e)

False
[]
Lỗi khi đọc đỉnh stack rỗng: list index out of range


### 2.4. `"".join(list_of_chars)` — ghép list ký tự thành chuỗi

Cuối cùng stack là một **list ký tự**; đề yêu cầu trả về **chuỗi**. `"".join(stack)` nối các phần tử bằng chuỗi rỗng ở giữa. Chi phí `O(n)`.

Đừng nối chuỗi bằng `+=` trong vòng lặp lớn — mỗi lần có thể tạo chuỗi mới.

In [5]:
chars = ["c", "a"]
print("".join(chars))      # ca
print("-".join(chars))     # c-a
print("".join([]))         # chuỗi rỗng ''
print(repr("".join([])))

ca
c-a

''


## 3. Cách 1 — Trực giác: quét và xóa nhiều lượt (mô phỏng đúng đề)

Ý tưởng tự nhiên nhất: làm đúng như đề nói.

1. Quét chuỗi từ trái sang phải, gặp cặp `s[i] == s[i+1]` thì xóa cặp đó.
2. Lặp lại toàn bộ cho tới khi một lượt quét **không xóa được gì**.

Ta dùng biến cờ `changed` để biết lượt vừa rồi có xóa không.

In [6]:
def remove_duplicates_brute(s: str) -> str:
    changed = True
    while changed:
        changed = False
        for i in range(len(s) - 1):          # đọc s[i+1] nên dừng ở len-2
            if s[i] == s[i + 1]:
                s = s[:i] + s[i + 2:]        # tạo chuỗi mới, bỏ 2 ký tự
                changed = True
                break                        # chuỗi đã đổi -> quét lại từ đầu
    return s

print(remove_duplicates_brute("abbaca"))   # ca
print(remove_duplicates_brute("azxxzy"))   # ay
print(repr(remove_duplicates_brute("aa"))) # ''

ca
ay
''


### Vì sao `range(len(s) - 1)`?

Trong vòng lặp ta đọc `s[i + 1]`. Nếu `i` chạy tới chỉ số cuối `len(s) - 1` thì `s[i + 1]` vượt biên. Vì thế `i` chỉ chạy tới `len(s) - 2`, tức `range(len(s) - 1)`.

In [7]:
s = "abb"
for i in range(len(s) - 1):
    print(i, s[i], "so với", s[i + 1], "->", s[i] == s[i + 1])

0 a so với b -> False
1 b so với b -> True


### Cách 2 — Biến thể trực giác: `str.replace` lặp

Chỉ có 26 chữ cái, nên các "cặp" có thể là `"aa"`, `"bb"`, …, `"zz"`. Ta lặp: thay mọi cặp bằng chuỗi rỗng, tới khi chuỗi không đổi nữa.

`s.replace(old, new)` thay **tất cả** lần xuất hiện **không chồng nhau** của `old`. Ví dụ `"aaa".replace("aa", "")` → `"a"`.

In [8]:
print("aaa".replace("aa", ""))      # a
print("abba".replace("bb", ""))     # aa  (lượt sau mới xóa "aa")

import string
pairs = [c * 2 for c in string.ascii_lowercase]   # ['aa', 'bb', ..., 'zz']
print(pairs[:5], "...", len(pairs), "cặp")

def remove_duplicates_replace(s: str) -> str:
    prev = None
    while prev != s:
        prev = s
        for p in pairs:
            s = s.replace(p, "")
    return s

print(remove_duplicates_replace("abbaca"), remove_duplicates_replace("azxxzy"))

a
aa
['aa', 'bb', 'cc', 'dd', 'ee'] ... 26 cặp
ca ay


### Nút thắt: vì sao cách xóa nhiều lượt chậm?

Xét chuỗi kiểu `"abcde...edcba"` (palindrome độ dài chẵn): mỗi lượt chỉ xóa được **đúng một cặp ở giữa**, rồi cặp mới lại lộ ra.

- Số lượt xóa ≈ `n / 2`.
- Mỗi lượt quét lại từ đầu + tạo chuỗi mới tốn `O(n)`.
- Tổng: `O(n²)`. Với `n = 10^5` → khoảng `10^10` thao tác: **quá chậm**.

Ta đếm số phép so sánh để thấy tận mắt.

In [9]:
def count_brute_work(s: str) -> int:
    work = 0
    changed = True
    while changed:
        changed = False
        for i in range(len(s) - 1):
            work += 1
            if s[i] == s[i + 1]:
                s = s[:i] + s[i + 2:]
                changed = True
                break
    return work

for half in [10, 100, 1000]:
    letters = "".join(chr(ord("a") + i % 26) for i in range(half))
    worst = letters + letters[::-1]          # dạng ...cba|abc... đối xứng
    print(f"n = {len(worst):5d} -> số phép so sánh ≈ {count_brute_work(worst)}")

n =    20 -> số phép so sánh ≈ 55
n =   200 -> số phép so sánh ≈ 5050
n =  2000 -> số phép so sánh ≈ 500500


Số phép so sánh tăng **gấp ~100 lần** khi `n` tăng 10 lần → đúng dáng `O(n²)`.

**Câu hỏi then chốt:** khi xóa một cặp, ký tự nào có thể tạo cặp mới? Chỉ có **ký tự ngay bên trái** cặp vừa xóa và **ký tự ngay bên phải**. Phần còn lại của chuỗi bên trái **đã ổn định** (không còn cặp), không cần quét lại!

## 4. Insight — giữ "phần đã ổn định" trong một stack

Đọc từng ký tự `ch` từ trái sang phải và giữ `stack` = **kết quả rút gọn của phần đã đọc**.

- Phần trong stack **không có cặp kề nhau giống nhau** (đã ổn định).
- Khi đọc `ch` mới, nó chỉ có thể "va chạm" với **ký tự cuối** của phần ổn định, tức `stack[-1]`.
  - Nếu `stack[-1] == ch` → cặp này biến mất: `stack.pop()` (và **không** thêm `ch`).
  - Ngược lại → `stack.append(ch)`.

Sau khi `pop`, đỉnh mới lộ ra — chính là "ký tự bên trái" có thể va chạm với ký tự tiếp theo. Phản ứng dây chuyền được xử lý **tự động**, không cần quét lại!

Hình dung như trò **xếp gạch kiểu Tetris/Zuma**: viên mới rơi xuống, nếu trùng màu viên trên cùng thì cả hai nổ.

In [10]:
# Thí nghiệm nhỏ: chỉ một bước quyết định
stack = ["a", "b"]
ch = "b"
if stack and stack[-1] == ch:
    stack.pop()
else:
    stack.append(ch)
print(stack)     # ['a']  -> "bb" đã nổ, đỉnh mới là 'a'

ch = "a"
if stack and stack[-1] == ch:
    stack.pop()
else:
    stack.append(ch)
print(stack)     # []     -> dây chuyền: "aa" nổ tiếp

['a']
[]


## 5. Cách 3 (lời giải chính) — Stack, ghép từng đoạn

**Đoạn 1 — khởi tạo:** `stack = []` (chưa đọc gì, phần ổn định rỗng).

**Đoạn 2 — duyệt:** `for ch in s:` đọc lần lượt từng ký tự.

**Đoạn 3 — quyết định:** nổ (`pop`) hoặc xếp lên (`append`).

**Đoạn 4 — trả kết quả:** `"".join(stack)`.

Ghép lại:

In [11]:
def remove_duplicates_stack(s: str) -> str:
    stack = []                         # phần đã ổn định
    for ch in s:
        if stack and stack[-1] == ch:  # va chạm với đỉnh
            stack.pop()                # cặp nổ
        else:
            stack.append(ch)           # xếp lên
    return "".join(stack)

print(remove_duplicates_stack("abbaca"))       # ca
print(remove_duplicates_stack("azxxzy"))       # ay
print(repr(remove_duplicates_stack("abba")))   # ''
print(remove_duplicates_stack("aaa"))          # a

ca
ay
''
a


## 6. Trace từng bước với `s = "abbaca"`

| Bước | `ch` | Đỉnh trước | Điều kiện `stack[-1] == ch` | Hành động | Stack sau |
|---:|:---:|:---:|:---:|---|---|
| 1 | a | (rỗng) | — | append | `[a]` |
| 2 | b | a | False | append | `[a, b]` |
| 3 | b | b | **True** | pop | `[a]` |
| 4 | a | a | **True** | pop | `[]` |
| 5 | c | (rỗng) | — | append | `[c]` |
| 6 | a | c | False | append | `[c, a]` |

Kết quả `"ca"`. Bước 4 chính là "phản ứng dây chuyền": sau khi `bb` nổ, hai chữ `a` gặp nhau.

In [12]:
def trace_remove_duplicates(s: str) -> str:
    stack = []
    for step, ch in enumerate(s, start=1):
        top = stack[-1] if stack else None
        if stack and stack[-1] == ch:
            stack.pop()
            action = "POP   (cặp nổ)"
        else:
            stack.append(ch)
            action = "APPEND"
        print(f"bước {step}: ch={ch!r}, đỉnh trước={top!r:5} -> {action:15} stack={stack}")
    result = "".join(stack)
    print("Kết quả:", repr(result))
    return result

trace_remove_duplicates("abbaca")

bước 1: ch='a', đỉnh trước=None  -> APPEND          stack=['a']
bước 2: ch='b', đỉnh trước='a'   -> APPEND          stack=['a', 'b']
bước 3: ch='b', đỉnh trước='b'   -> POP   (cặp nổ)  stack=['a']
bước 4: ch='a', đỉnh trước='a'   -> POP   (cặp nổ)  stack=[]
bước 5: ch='c', đỉnh trước=None  -> APPEND          stack=['c']
bước 6: ch='a', đỉnh trước='c'   -> APPEND          stack=['c', 'a']
Kết quả: 'ca'


'ca'

### Giải thích cú pháp mới trong hàm trace

- `enumerate(s, start=1)` trả về từng cặp `(số thứ tự, ký tự)`, đánh số bắt đầu từ 1.
- `x if điều_kiện else y` là **biểu thức điều kiện** (ternary): `stack[-1] if stack else None` lấy đỉnh nếu có, ngược lại `None`.
- Trong f-string, `{ch!r}` in dạng `repr` (có dấu nháy), `{action:15}` canh rộng 15 ký tự.

Trace thêm ví dụ thứ hai, có dây chuyền dài hơn:

In [13]:
trace_remove_duplicates("azxxzy")

bước 1: ch='a', đỉnh trước=None  -> APPEND          stack=['a']
bước 2: ch='z', đỉnh trước='a'   -> APPEND          stack=['a', 'z']
bước 3: ch='x', đỉnh trước='z'   -> APPEND          stack=['a', 'z', 'x']
bước 4: ch='x', đỉnh trước='x'   -> POP   (cặp nổ)  stack=['a', 'z']
bước 5: ch='z', đỉnh trước='z'   -> POP   (cặp nổ)  stack=['a']
bước 6: ch='y', đỉnh trước='a'   -> APPEND          stack=['a', 'y']
Kết quả: 'ay'


'ay'

## 7. Cách 4 — Hai con trỏ ghi đè (two pointers, mô phỏng stack trên mảng)

Ý tưởng: dùng chính mảng ký tự làm stack.

- `res` = list ký tự của `s`; `top` = **số phần tử đang nằm trong stack** (đỉnh là `res[top - 1]`).
- Con trỏ đọc `i` đi qua từng ký tự; con trỏ ghi `top` chỉ chỗ sẽ ghi tiếp.
- Ghi `res[top] = s[i]`; nếu `top > 0` và `res[top] == res[top - 1]` thì "nổ" bằng `top -= 1`, ngược lại `top += 1`.

Cách này giống hệt stack nhưng không cần `append/pop` — rất quen thuộc trong C/Java, và là bài tập tốt về con trỏ.

In [14]:
def remove_duplicates_two_pointers(s: str) -> str:
    res = list(s)
    top = 0                         # số phần tử trong "stack"
    for i in range(len(s)):
        res[top] = s[i]             # ghi tạm ký tự mới lên đỉnh
        if top > 0 and res[top - 1] == res[top]:
            top -= 1                # nổ: bỏ luôn ký tự dưới đỉnh
        else:
            top += 1                # giữ lại
    return "".join(res[:top])

print(remove_duplicates_two_pointers("abbaca"))
print(remove_duplicates_two_pointers("azxxzy"))
print(repr(remove_duplicates_two_pointers("aabb")))

ca
ay
''


In [15]:
def trace_two_pointers(s: str) -> None:
    res = list(s)
    top = 0
    for i in range(len(s)):
        res[top] = s[i]
        if top > 0 and res[top - 1] == res[top]:
            top -= 1
            note = "nổ  -> top giảm"
        else:
            top += 1
            note = "giữ -> top tăng"
        print(f"i={i}, s[i]={s[i]}, {note:16} top={top}, stack={res[:top]}")
    print("Kết quả:", "".join(res[:top]))

trace_two_pointers("abbaca")

i=0, s[i]=a, giữ -> top tăng  top=1, stack=['a']
i=1, s[i]=b, giữ -> top tăng  top=2, stack=['a', 'b']
i=2, s[i]=b, nổ  -> top giảm  top=1, stack=['a']
i=3, s[i]=a, nổ  -> top giảm  top=0, stack=[]
i=4, s[i]=c, giữ -> top tăng  top=1, stack=['c']
i=5, s[i]=a, giữ -> top tăng  top=2, stack=['c', 'a']
Kết quả: ca


## 8. Code nộp LeetCode

### Bản chính — Stack (khuyên dùng)

In [16]:
class Solution:
    def removeDuplicates(self, s: str) -> str:
        stack = []
        for ch in s:
            if stack and stack[-1] == ch:
                stack.pop()
            else:
                stack.append(ch)
        return "".join(stack)

sol = Solution()
assert sol.removeDuplicates("abbaca") == "ca"
assert sol.removeDuplicates("azxxzy") == "ay"
print("Solution (stack) PASS")

Solution (stack) PASS


### Bản hai con trỏ

In [17]:
class SolutionTwoPointers:
    def removeDuplicates(self, s: str) -> str:
        res = list(s)
        top = 0
        for ch in s:
            res[top] = ch
            if top > 0 and res[top - 1] == ch:
                top -= 1
            else:
                top += 1
        return "".join(res[:top])

assert SolutionTwoPointers().removeDuplicates("abbaca") == "ca"
assert SolutionTwoPointers().removeDuplicates("azxxzy") == "ay"
print("SolutionTwoPointers PASS")

SolutionTwoPointers PASS


## 9. Vì sao lời giải stack đúng?

**Invariant (bất biến):** sau khi đọc xong `k` ký tự đầu, `stack` chính là **kết quả rút gọn hoàn toàn** của `s[:k]` — và trong stack không có hai ký tự kề nhau giống nhau.

- **Khởi đầu** (`k = 0`): stack rỗng = kết quả của chuỗi rỗng. Đúng.
- **Bước chuyển:** giả sử invariant đúng với `s[:k]`, đọc `ch = s[k]`.
  - Phần trong stack không có cặp nào, nên cặp mới (nếu có) **chỉ có thể** là (đỉnh, `ch`).
  - Nếu đỉnh == `ch`: xóa cặp. Phần còn lại vẫn là stack cũ bỏ đỉnh — không có cặp (vì là con của dãy không có cặp). Invariant giữ.
  - Nếu khác: thêm `ch` không tạo cặp nào. Invariant giữ.
- **Kết thúc** (`k = n`): stack là kết quả rút gọn của toàn bộ `s`.

Vì sao **đáp án duy nhất**? Có thể hình dung mỗi ký tự là một "dấu ngoặc" khớp với ký tự giống nó; việc xóa giống khử cặp ngoặc, thứ tự khử không làm đổi kết quả cuối (tính chất hợp lưu — confluence). Ta không cần chứng minh chặt; chỉ cần thấy stack mô phỏng **một** thứ tự xóa hợp lệ, nên cho đúng đáp án duy nhất ấy.

## 10. Độ phức tạp

Gọi `n = len(s)`.

| Cách | Time | Space phụ | Ghi chú |
|---|---|---|---|
| 1. Xóa nhiều lượt (brute) | `O(n²)` | `O(n)` (chuỗi mới mỗi lần) | quá chậm với `n = 10^5` |
| 2. `replace` lặp | `O(26 · n · số lượt)` → worst `O(n²)` | `O(n)` | ngắn nhưng worst vẫn chậm |
| 3. Stack | `O(n)` | `O(n)` | mỗi ký tự push ≤ 1 lần, pop ≤ 1 lần |
| 4. Hai con trỏ | `O(n)` | `O(n)` (list ký tự, vì chuỗi Python bất biến) | trong C++ có thể `O(1)` phụ |

**Phân tích khấu hao (amortized):** trong cách stack, một vòng `for` có thể `pop`, nhưng tổng số `pop` không vượt tổng số `append` ≤ `n`, nên tổng công việc `O(n)`.

In [18]:
import time
n = 100_000
half = "".join(chr(ord("a") + i % 26) for i in range(n // 2))
big = half + half[::-1]           # worst case cho brute force, rút gọn thành ''
t0 = time.perf_counter()
r = remove_duplicates_stack(big)
t1 = time.perf_counter()
print(f"stack với n={len(big)}: kết quả rỗng? {r == ''}, thời gian {t1 - t0:.4f}s")

stack với n=100000: kết quả rỗng? True, thời gian 0.0042s


## 11. Bộ test

Các nhóm test:
1. Ví dụ đề bài.
2. Biên: một ký tự, hai ký tự giống/khác.
3. Xóa sạch: `"abba"`, `"aabbcc"`.
4. Số lẻ ký tự giống nhau: `"aaa"` → `"a"`, `"aaaaa"` → `"a"`.
5. Dây chuyền dài: palindrome chẵn.
6. Đối chiếu ngẫu nhiên 4 cách với nhau.

In [19]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"{name:28} PASS -> {actual!r}")

cases = [
    ("đề 1", "abbaca", "ca"),
    ("đề 2", "azxxzy", "ay"),
    ("1 ký tự", "a", "a"),
    ("2 ký tự giống", "aa", ""),
    ("2 ký tự khác", "ab", "ab"),
    ("xóa sạch abba", "abba", ""),
    ("xóa sạch aabbcc", "aabbcc", ""),
    ("3 chữ a", "aaa", "a"),
    ("5 chữ a", "aaaaa", "a"),
    ("dây chuyền", "abccba", ""),
    ("không có cặp", "abcabc", "abcabc"),
]
for name, s, exp in cases:
    check(name, remove_duplicates_stack(s), exp)
    check(name + " (2 con trỏ)", remove_duplicates_two_pointers(s), exp)

đề 1                         PASS -> 'ca'
đề 1 (2 con trỏ)             PASS -> 'ca'
đề 2                         PASS -> 'ay'
đề 2 (2 con trỏ)             PASS -> 'ay'
1 ký tự                      PASS -> 'a'
1 ký tự (2 con trỏ)          PASS -> 'a'
2 ký tự giống                PASS -> ''
2 ký tự giống (2 con trỏ)    PASS -> ''
2 ký tự khác                 PASS -> 'ab'
2 ký tự khác (2 con trỏ)     PASS -> 'ab'
xóa sạch abba                PASS -> ''
xóa sạch abba (2 con trỏ)    PASS -> ''
xóa sạch aabbcc              PASS -> ''
xóa sạch aabbcc (2 con trỏ)  PASS -> ''
3 chữ a                      PASS -> 'a'
3 chữ a (2 con trỏ)          PASS -> 'a'
5 chữ a                      PASS -> 'a'
5 chữ a (2 con trỏ)          PASS -> 'a'
dây chuyền                   PASS -> ''
dây chuyền (2 con trỏ)       PASS -> ''
không có cặp                 PASS -> 'abcabc'
không có cặp (2 con trỏ)     PASS -> 'abcabc'


In [20]:
import random
random.seed(1047)
for _ in range(2000):
    length = random.randint(1, 12)
    s = "".join(random.choice("abc") for _ in range(length))   # ít chữ -> nhiều cặp
    a = remove_duplicates_brute(s)
    b2 = remove_duplicates_replace(s)
    c = remove_duplicates_stack(s)
    d = remove_duplicates_two_pointers(s)
    assert a == b2 == c == d, (s, a, b2, c, d)
print("2000 test ngẫu nhiên: 4 cách cho cùng kết quả")

2000 test ngẫu nhiên: 4 cách cho cùng kết quả


## 12. Lỗi người mới hay gặp

### Lỗi 1: Chỉ quét **một lượt** và so sánh với ký tự kề trong chuỗi gốc

```python
def wrong_one_pass(s):
    res = []
    i = 0
    while i < len(s):
        if i + 1 < len(s) and s[i] == s[i + 1]:
            i += 2          # bỏ cặp
        else:
            res.append(s[i]); i += 1
    return "".join(res)
```

Sai vì bỏ qua phản ứng dây chuyền: `"abbaca"` → `"aaca"` chứ không ra `"ca"`. **Sửa:** so với `stack[-1]` (kết quả đã rút gọn), không phải `s[i+1]` (chuỗi gốc).

In [21]:
def wrong_one_pass(s):
    res = []
    i = 0
    while i < len(s):
        if i + 1 < len(s) and s[i] == s[i + 1]:
            i += 2
        else:
            res.append(s[i]); i += 1
    return "".join(res)

print("sai :", wrong_one_pass("abbaca"))
print("đúng:", remove_duplicates_stack("abbaca"))

sai : aaca
đúng: ca


### Lỗi 2: Xóa **mọi** ký tự lặp liền nhau (cả nhóm) thay vì từng cặp

Có người nghĩ `"aaa"` → `""` vì "a bị lặp". Sai: đề xóa theo **cặp**, `"aaa"` còn dư một `a`.

### Lỗi 3: Quên kiểm tra stack rỗng

```python
if stack[-1] == ch:   # IndexError ở ký tự đầu tiên!
```
**Sửa:** `if stack and stack[-1] == ch:`.

### Lỗi 4: Vừa `pop` vừa `append`

```python
if stack and stack[-1] == ch:
    stack.pop()
stack.append(ch)      # sai: ch phải biến mất cùng cặp
```

In [22]:
def wrong_pop_then_append(s):
    stack = []
    for ch in s:
        if stack and stack[-1] == ch:
            stack.pop()
        stack.append(ch)        # luôn append -> sai
    return "".join(stack)

print("sai :", wrong_pop_then_append("abbaca"))   # không nổ được cặp nào
print("đúng:", remove_duplicates_stack("abbaca"))

sai : abaca
đúng: ca


### Lỗi 5: Nối chuỗi `result += ch` / `result = result[:-1]`

Về logic thì đúng, nhưng mỗi lần cắt `result[:-1]` tạo chuỗi mới `O(n)` → tổng có thể `O(n²)`. Dùng list làm stack rồi `join` một lần ở cuối.

## 13. Mẹo nhận dạng pattern "stack khử cặp"

Nghĩ ngay tới **stack** khi thấy:
- "xóa/khử **cặp kề nhau**" và việc xóa có thể làm lộ ra cặp mới (dây chuyền).
- "dấu ngoặc hợp lệ", "backspace `#`", "va chạm thiên thạch" (asteroid collision).
- Quyết định cho phần tử mới **chỉ phụ thuộc phần tử gần nhất còn sống**.

Câu thần chú: *"Phần bên trái đã ổn định — chỉ đỉnh stack mới có thể tương tác với phần tử mới."*

| Bài | Liên hệ |
|---|---|
| 20. Valid Parentheses | khử cặp `(` `)` bằng stack |
| 844. Backspace String Compare | `#` = pop |
| 1209. Remove All Adjacent Duplicates in String II | khử khi đủ `k` ký tự giống nhau |
| 1544. Make The String Great | khử cặp chữ hoa/thường cùng chữ |
| 735. Asteroid Collision | va chạm với đỉnh stack |

## 14. Biến thể

### Biến thể A — Backspace: `#` xóa ký tự trước nó
Cùng khuôn stack: gặp `#` thì `pop` (nếu có), ngược lại `append`.

In [23]:
def apply_backspace(s: str) -> str:
    stack = []
    for ch in s:
        if ch == "#":
            if stack:
                stack.pop()
        else:
            stack.append(ch)
    return "".join(stack)

print(apply_backspace("ab#c"))    # ac
print(apply_backspace("a##c"))    # c

ac
c


### Biến thể B — Xóa khi có `k` ký tự giống nhau liền kề (LeetCode 1209)

Stack lưu cặp `[ký tự, số lần liên tiếp]`. Khi số lần đạt `k` thì `pop` cả nhóm. Với `k = 2` ta quay về bài hiện tại.

In [24]:
def remove_k_duplicates(s: str, k: int) -> str:
    stack = []                       # mỗi phần tử: [ch, count]
    for ch in s:
        if stack and stack[-1][0] == ch:
            stack[-1][1] += 1
            if stack[-1][1] == k:
                stack.pop()
        else:
            stack.append([ch, 1])
    return "".join(ch * cnt for ch, cnt in stack)

print(remove_k_duplicates("deeedbbcccbdaa", 3))   # aa
print(remove_k_duplicates("abbaca", 2))           # ca (giống bài 1047)

aa
ca


## 15. Bài tập tự luyện (tăng dần)

> Làm trong các cell TODO. Đừng kéo xuống phần đáp án trước khi thử!

**Bài 1 (tính tay):** Với `s = "aabccbd"`, hãy viết bảng trace stack và cho biết kết quả. (Kiểm tra bằng `trace_remove_duplicates` sau khi tự làm.)

**Bài 2 (điền chỗ trống):** Hoàn thiện `ex_remove_duplicates` theo khung TODO.

**Bài 3 (biến thể):** `ex_make_good(s)` — xóa cặp kề nhau là **cùng một chữ nhưng khác hoa/thường** (`"aA"`, `"Bb"`), lặp tới khi hết (LeetCode 1544).

**Bài 4 (đếm):** `ex_count_removals(s)` — trả về **số cặp** đã bị xóa trong quá trình rút gọn.

**Bài 5 (mini challenge):** `ex_backspace_compare(s, t)` — `True` nếu hai chuỗi bằng nhau sau khi áp dụng backspace `#` (LeetCode 844).

In [25]:
def ex_remove_duplicates(s: str) -> str:
    stack = []
    for ch in s:
        # TODO 1: nếu stack khác rỗng và đỉnh bằng ch -> pop
        # TODO 2: ngược lại -> append
        pass
    # TODO 3: trả về chuỗi ghép từ stack
    pass

# Tự làm xong rồi mới mở test:
# assert ex_remove_duplicates("abbaca") == "ca"
# assert ex_remove_duplicates("aabccbd") == "d"

In [26]:
def ex_make_good(s: str) -> str:
    # TODO: stack; hai ký tự "khử nhau" khi khác nhau nhưng lower() giống nhau
    pass

# assert ex_make_good("leEeetcode") == "leetcode"
# assert ex_make_good("abBAcC") == ""

In [27]:
def ex_count_removals(s: str) -> int:
    # TODO: giống lời giải stack, thêm một biến đếm mỗi lần pop
    pass

# assert ex_count_removals("abbaca") == 2
# assert ex_count_removals("abc") == 0

In [28]:
def ex_backspace_compare(s: str, t: str) -> bool:
    # TODO: viết hàm phụ build(x) dùng stack, rồi so sánh build(s) == build(t)
    pass

# assert ex_backspace_compare("ab#c", "ad#c") == True
# assert ex_backspace_compare("a#c", "b") == False

## 16. Gợi ý 3 mức

**Bài 1:** (1) Đi từng ký tự, chỉ nhìn đỉnh. (2) `aa` nổ ngay; `b c c b` → `cc` nổ rồi `bb` nổ. (3) Còn lại đúng một ký tự cuối.

**Bài 2:** (1) Đúng như mục 6. (2) `stack and stack[-1] == ch`. (3) `return "".join(stack)`.

**Bài 3:** (1) Vẫn là khử cặp với đỉnh stack, chỉ đổi điều kiện "khử". (2) Hai chữ `x`, `y` khử nhau khi `x != y and x.lower() == y.lower()`. (3) `if stack and stack[-1] != ch and stack[-1].lower() == ch.lower(): stack.pop()`.

**Bài 4:** (1) Mỗi lần nổ = một cặp. (2) Khai báo `count = 0` trước vòng lặp. (3) Trong nhánh `pop` thêm `count += 1`.

**Bài 5:** (1) Rút gọn từng chuỗi rồi so sánh. (2) Hàm phụ `build` dùng stack, `#` thì pop nếu có. (3) `return build(s) == build(t)`.

## 17. Đáp án tham khảo

Nên tự làm trước rồi mới xem phần này. Đáp án bài 1: `"aabccbd"` → stack `[a]`, `[]`, `[b]`, `[b,c]`, `[b]`, `[]`, `[d]` → `"d"`.

In [29]:
def ans_remove_duplicates(s: str) -> str:
    stack = []
    for ch in s:
        if stack and stack[-1] == ch:
            stack.pop()
        else:
            stack.append(ch)
    return "".join(stack)

def ans_make_good(s: str) -> str:
    stack = []
    for ch in s:
        if stack and stack[-1] != ch and stack[-1].lower() == ch.lower():
            stack.pop()
        else:
            stack.append(ch)
    return "".join(stack)

def ans_count_removals(s: str) -> int:
    stack, count = [], 0
    for ch in s:
        if stack and stack[-1] == ch:
            stack.pop()
            count += 1
        else:
            stack.append(ch)
    return count

def ans_backspace_compare(s: str, t: str) -> bool:
    def build(x: str) -> str:
        st = []
        for ch in x:
            if ch == "#":
                if st:
                    st.pop()
            else:
                st.append(ch)
        return "".join(st)
    return build(s) == build(t)

assert ans_remove_duplicates("abbaca") == "ca"
assert ans_remove_duplicates("aabccbd") == "d"
assert ans_make_good("leEeetcode") == "leetcode"
assert ans_make_good("abBAcC") == ""
assert ans_make_good("s") == "s"
assert ans_count_removals("abbaca") == 2
assert ans_count_removals("abc") == 0
assert ans_backspace_compare("ab#c", "ad#c") == True
assert ans_backspace_compare("a#c", "b") == False
assert ans_backspace_compare("a##c", "#a#c") == True
trace_remove_duplicates("aabccbd")
print("Đáp án tham khảo PASS toàn bộ test!")

bước 1: ch='a', đỉnh trước=None  -> APPEND          stack=['a']
bước 2: ch='a', đỉnh trước='a'   -> POP   (cặp nổ)  stack=[]
bước 3: ch='b', đỉnh trước=None  -> APPEND          stack=['b']
bước 4: ch='c', đỉnh trước='b'   -> APPEND          stack=['b', 'c']
bước 5: ch='c', đỉnh trước='c'   -> POP   (cặp nổ)  stack=['b']
bước 6: ch='b', đỉnh trước='b'   -> POP   (cặp nổ)  stack=[]
bước 7: ch='d', đỉnh trước=None  -> APPEND          stack=['d']
Kết quả: 'd'
Đáp án tham khảo PASS toàn bộ test!


## 18. Template nhớ nhanh + checklist submit

```python
class Solution:
    def removeDuplicates(self, s: str) -> str:
        stack = []
        for ch in s:
            if stack and stack[-1] == ch:   # va chạm với đỉnh
                stack.pop()                 # cặp nổ
            else:
                stack.append(ch)            # xếp lên
        return "".join(stack)
```

**Khuôn "stack khử cặp":** `for x in data: if stack and cancels(stack[-1], x): pop else: push`.

**Checklist trước khi submit:**
- [ ] So sánh với `stack[-1]` (kết quả đã rút gọn), **không** với `s[i+1]`.
- [ ] Có kiểm tra `stack` khác rỗng trước khi đọc `stack[-1]`.
- [ ] Nhánh nổ **không** append ký tự hiện tại.
- [ ] Trả về `str` bằng `"".join(stack)`, không trả list.
- [ ] Test `"aaa"` → `"a"`, `"abba"` → `""`.
- [ ] Độ phức tạp `O(n)` time, `O(n)` space.